# Lab 01: The Agent Loop

**Goal:** understand what an agent *is* in Strands by seeing the loop work: model → tool call → tool result → model, until it stops.

Course module: `samples/01-agent-loop`

```
 user prompt ─► [ model ] ──text──► done
                   │ ▲
          tool_use │ │ tool_result
                   ▼ │
                [ tools ]
```
An agent is a model, a system prompt, and tools, plus a **harness** that runs this loop and keeps the conversation history.

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

## 1. The simplest agent and what it returns

In [ ]:
from strands import Agent

agent = Agent(model=get_model(), callback_handler=None)  # None = don't stream; we'll print ourselves
result = agent("In one sentence: what is the capital of Australia?")

print("Answer:     ", str(result).strip())
print("stop_reason:", result.stop_reason)   # 'end_turn' = model finished on its own

`AgentResult` also carries metrics: token usage, loop cycles, and latency.

In [ ]:
summary = result.metrics.get_summary()
print("cycles:", summary["total_cycles"])
print("tokens:", summary["accumulated_usage"])

## 2. Conversation history is the agent's state
Each call appends to `agent.messages`. The next call sends the whole history, which is how the agent "remembers".

In [ ]:
_ = agent("What's the population of that city, roughly?")  # 'that city' only resolves from history
print()
for m in agent.messages:
    text = " ".join(b.get("text", "") for b in m["content"]).strip().replace("\n", " ")
    print(f"{m['role']:9} | {text[:100]}")

## 3. Give the agent a tool
`@tool` turns a Python function into a tool. The **docstring and type hints become the schema** the model sees, so write them for the model.

In [ ]:
from strands import tool

@tool
def query_product_database(query: str) -> str:
    """Query the internal product database for inventory and pricing information.

    Args:
        query: Search query for products (e.g., "wireless headphones", "USB-C hub")
    """
    products = {
        "wireless headphones": "SKU-WH100: Wireless Headphones Pro - $79.99, 142 in stock, 4.5★",
        "usb-c hub": "SKU-UC200: USB-C Hub 7-in-1 - $45.00, 89 in stock, 4.2★",
        "mechanical keyboard": "SKU-MK300: Mechanical Keyboard RGB - $149.99, 23 in stock, 4.8★",
        "noise cancelling": "SKU-NC400: Noise Cancelling Earbuds - $129.99, 67 in stock, 4.6★",
    }
    key = query.lower()
    matches = [v for k, v in products.items() if k in key]
    return "\n".join(matches) or f"No products found matching '{query}'. Available: {', '.join(products)}"

import json
print(json.dumps(query_product_database.tool_spec, indent=2))

In [ ]:
SYSTEM_PROMPT = """You are a product analyst. Use query_product_database to answer
questions about our catalog. Be brief."""

agent = Agent(model=get_model(), tools=[query_product_database], system_prompt=SYSTEM_PROMPT)
result = agent("Do we have wireless headphones and a mechanical keyboard in stock? Which is cheaper?")

## 4. Watch the loop in the message history
The tool round trip shows up as `toolUse` (from the assistant) and `toolResult` (sent back as the user role).

In [ ]:
for i, m in enumerate(agent.messages):
    for block in m["content"]:
        if "text" in block:
            kind, detail = "text", block["text"][:80].replace("\n", " ")
        elif "toolUse" in block:
            kind, detail = "toolUse", f'{block["toolUse"]["name"]}({block["toolUse"]["input"]})'
        elif "toolResult" in block:
            kind, detail = "toolResult", str(block["toolResult"]["content"])[:80].replace("\n", " ")
        else:
            continue
        print(f"[{i}] {m['role']:9} {kind:10} {detail}")

print("\ncycles:", result.metrics.get_summary()["total_cycles"])

## 5. Call a tool directly (no model)
Useful for testing tools. `record_direct_tool_call=True` (the default) also writes the call into history.

In [ ]:
direct = agent.tool.query_product_database(query="usb-c hub")
print(direct["status"], "→", direct["content"][0]["text"])

## 🧪 Exercise
1. Add a second tool `check_competitor_price(product: str) -> str` that returns hard-coded competitor prices.
2. Ask the agent: *"Are we priced competitively on noise cancelling earbuds?"*
3. Inspect `agent.messages`. Did it call both tools? In one cycle or several?

**Next:** [Lab 02: Model Providers](02-model-providers.ipynb)

In [ ]:
# your code here